### RAG 생성해보기

### 1. Loader

In [19]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/가상Tech_업무가이드.pdf") # 로더 설정 

In [20]:
docs = loader.load() # pdf 로더

len(docs) # 로드 된 문서 수 확인 (pdf 페이지 수와 동일해야 함)

9

In [21]:
docs[0] # 첫 번째 문서 확인 (pdf 첫 페이지 내용과 동일해야 함)

Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'author': '가상Tech People Operations', 'keywords': '', 'moddate': '2026-09-26T11:41:50+09:00', 'subject': 'RAG 실습용 가상 사내 업무 가이드', 'title': '가상Tech 업무 가이드', 'trapped': '/False', 'source': 'data/가상Tech_업무가이드.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='VIRTUALTECH · INTERNAL HANDBOOK\n가상Tech\n업무 가이드\n함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지\n구성원이 매일 참고하는 실무 기준\n 문서 버전\n 시행일\n 문서 등급\n 문의\nv1.0\n2026-09-01\n사내 공개\nPeople Operations\n이 문서의 목적\n가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.\n실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.')

### 2. TextSplitter
- 문자 단위로 재귀적으로 분할하기

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 청크 하나당 최대 글자 수
    chunk_overlap=50,    # 앞 뒤 청크끼리 겹치는 글자 수 (문맥 끊김 방지, 보통 10~20% 정도 겹치게 설정)
    length_function=len,          # 청크 길이를 잴 때 어떤 함수로 할지 (기본값이 이미 len)
    is_separator_regex=False,     # 구분자 리스트를 정규 표현식으로 처리할지 여부 False면 문자열 그대로 처리 (기본값이 이미 False)
)

split_docs = text_splitter.split_documents(docs)  # docs(현재 페이지 단위)를 더 작은 청크 단위로 분할

len(split_docs)  # 분할된 청크 개수 확인

18

In [23]:
split_docs[0]  # 분할된 첫 번째 청크 확인

Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'author': '가상Tech People Operations', 'keywords': '', 'moddate': '2026-09-26T11:41:50+09:00', 'subject': 'RAG 실습용 가상 사내 업무 가이드', 'title': '가상Tech 업무 가이드', 'trapped': '/False', 'source': 'data/가상Tech_업무가이드.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='VIRTUALTECH · INTERNAL HANDBOOK\n가상Tech\n업무 가이드\n함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지\n구성원이 매일 참고하는 실무 기준\n 문서 버전\n 시행일\n 문서 등급\n 문의\nv1.0\n2026-09-01\n사내 공개\nPeople Operations\n이 문서의 목적\n가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.\n실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.')

### 3. Embedding
- 여러 모델 중 openaiembeddings 사용

In [24]:
from langchain_teddynote import logging # 로깅 설정
from dotenv import load_dotenv # 환경변수 로드

load_dotenv() # .env 파일 로드
logging.langsmith("rag_01_0928") # 로깅 시작

LangSmith 추적을 시작합니다.
[프로젝트명]
rag_01_0928


In [25]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # 임베딩 모델 설정

In [26]:
sample_vector = embeddings.embed_query("이 문서는 어떤 내용을 다루나요?")  # 해당 쿼리 하나를 벡터로 변환해봐서 잘 되는지 테스트

print(len(sample_vector))   # 벡터 차원 수 확인 (text-embedding-3-small은 1536)
print(sample_vector[:5])    # 벡터 앞부분 일부만 확인

1536
[0.049835205078125, 0.054290771484375, 0.0008974075317382812, -0.002307891845703125, -0.00582122802734375]


### 4. VectorStore
- 로컬용인 chroma와 faiss 중 faiss 사용


In [27]:
from langchain_community.vectorstores import FAISS

# 그릇 만들기(FAISS 벡터 스토어 생성) + 청크 벡터화 + 벡터 스토어에 담기
db = FAISS.from_documents(documents=split_docs, embedding=embeddings) # 분할된 문서들을 벡터화해서 FAISS 벡터 스토어 생성


### 5. Retriever
- 가장 기본인 벡터 스토어 기반 리트리버
- 선택기준요약
    - 기본은 무조건 벡터스토어 Retriever — 대부분의 실습/소규모 프로젝트에 충분
    검색이 누락된다 (질문 표현과 문서 표현이 다름) → 다중 쿼리 or 앙상블(BM25+벡터)
    관련 없는 내용이 너무 많이 딸려온다 → 문서 압축기
    청크가 너무 작아 문맥이 부족하다 → 부모 문서 리트리버
    검색 결과 개수가 많고 LLM이 중간 내용을 놓친다 → 긴 문맥 재정렬

In [28]:
from langchain_community.vectorstores import FAISS

retriever = db.as_retriever() # FAISS 벡터 스토어를 검색 가능한 retriever로 변환

In [29]:
retriever.invoke("연차를 사용하려면 최소 며칠 전에 신청해야 하나요?")  # 문서 내용에 맞는 실제 질문으로 테스트
# 옵션없이 호출하면 기본값 k =4로 가장 유사한 4개의 청크를 반환


[Document(id='4eeb938a-ea44-472d-b96c-79686c806aca', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'author': '가상Tech People Operations', 'keywords': '', 'moddate': '2026-09-26T11:41:50+09:00', 'subject': 'RAG 실습용 가상 사내 업무 가이드', 'title': '가상Tech 업무 가이드', 'trapped': '/False', 'source': 'data/가상Tech_업무가이드.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}, page_content='• 문제는 사람보다 시스템에서 찾는다: 회고는 비난 없이 재발 방지에 집중한다.\n • 기본 공개, 필요한 경우 제한: 업무 정보는 사내 공개가 원칙이며 개인정보와 고객정보만 최소 권한으로 제한한다.\n \n1.3 근무 방식\n 항목\n 기준\n 세부 규칙\n출근\n유연근무\n08:00-11:00 사이 출근, 일 8시간 근무\n재택\n주 2회\n매주 월요일까지 캘린더에 ‘WFH’ 표시\n회의\n25분/50분\n안건 없는 회의는 거절 가능, 회의록은 24시간 내 등록\n집중시간\n14:00-16:00\n화·목은 사내 정기회의 생성 금지\n응답 기대\n업무시간 4시간\n긴급은 Slack #help-urgent와 전화 병행\n신입 첫날 체크\nHRIS 프로필 작성 → 보안 교육 수강 → MFA 등록 → 팀 채널 입장 → 버디와 30분\n온보딩. 모든 항목은 입사일 18:00까지 완료합니다.'),
 Document(id='993c2c82-08c8-4290-83b5-9fdd3ac2645c', metadata={'producer': 'ReportLab PDF 

### 6. Prompt
- ChatPromptTemplate + {context}/{question} 변수 사용
- 프롬프트 관련 기법 정리
    - 프롬프트 템플릿 만들기: {변수} 자리표시자를 넣은 템플릿 문자열 작성. RAG의 기본 구성 요소
    - ChatPromptTemplate: System/Human 등 역할(role)별로 메시지를 나눠 구성. ChatOpenAI처럼 대화형 모델에 쓸 때 사용
    - 부분 변수 활용(partial_variables): 템플릿의 일부 값을 미리 고정해둘 때 사용 (예: 항상 "한국어로 답하라", 오늘 날짜 등 고정값이 있을 때)
    - YAML 파일로부터 로드: 프롬프트를 코드가 아닌 별도 yaml 파일로 관리하고 싶을 때 사용. 여러 프롬프트를 버전 관리하거나 팀 단위로 공유할 때 유용
    - MessagePlaceholder: 이전 대화 기록(히스토리)을 프롬프트 중간에 끼워넣을 자리가 필요할 때 사용. 멀티턴 대화형 챗봇에서 사용
    - 퓨샷 프롬프트(Few-shot): "이런 질문엔 이렇게 답하라"는 예시를 프롬프트에 미리 넣어, 답변 형식/스타일을 일관되게 통제하고 싶을 때 사용
    - 예제 선택기(Example Selector): 퓨샷 예제가 많을 때, 매 질문마다 가장 유사한 예제만 골라서 넣고 싶을 때 사용

In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 사내 문서를 기반으로 답변하는 어시스턴트입니다. 주어진 context만 참고해서 답하세요."),
    ("human", "context: {context}\n\n질문: {question}")
])


In [31]:
prompt.invoke({"context": "샘플 컨텍스트", "question": "테스트 질문"})

ChatPromptValue(messages=[SystemMessage(content='당신은 사내 문서를 기반으로 답변하는 어시스턴트입니다. 주어진 context만 참고해서 답하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='context: 샘플 컨텍스트\n\n질문: 테스트 질문', additional_kwargs={}, response_metadata={})])

### 7. LLM
- ChatOpenAI 사용 (gpt-4o-mini)

In [32]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 답변 생성 모델, temperature=0으로 일관된 답변 유도

### 8. OutputParser
- StrOutputParser 사용 (AIMessage -> 순수 텍스트 추출)

In [33]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()  # LLM 응답(AIMessage)에서 순수 텍스트만 추출

### 9. Chain
- retriever + prompt(context/question 조합) + llm(답변 생성) + output_parser(텍스트 추출)를 LCEL(|)로 연결
- 질문 하나만 입력하면 retriever가 자동으로 context를 채워주도록 RunnablePassthrough 사용

In [37]:
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)  # 검색된 Document 리스트를 하나의 context 문자열로 합침


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)  # 질문 -> 검색 -> 프롬프트 -> LLM -> 텍스트 답변까지 이어지는 최종 RAG 체인

rag_chain.invoke("재택근무는 얼마나 자주 할 수 있나요?")  # 체인 동작 테스트

"재택근무는 주  2회 가능합니다. 매주 월요일까지 캘린더에 'WFH' 표시를 해야 합니다."

### 10. 질문 10개 테스트

In [36]:
print("[Q1] 가상Tech의 유연근무 출근 시간 기준은 어떻게 되나요?")
print("[A1]", rag_chain.invoke("가상Tech의 유연근무 출근 시간 기준은 어떻게 되나요?"), "\n")

print("[Q2] 재택근무는 일주일에 몇 번, 어떻게 신청하나요?")
print("[A2]", rag_chain.invoke("재택근무는 일주일에 몇 번, 어떻게 신청하나요?"), "\n")

print("[Q3] 회의 시간 기준(25분/50분)은 왜 정해져 있나요?")
print("[A3]", rag_chain.invoke("회의 시간 기준(25분/50분)은 왜 정해져 있나요?"), "\n")

print("[Q4] 신입사원 첫날에는 어떤 절차를 거쳐야 하나요?")
print("[A4]", rag_chain.invoke("신입사원 첫날에는 어떤 절차를 거쳐야 하나요?"), "\n")

print("[Q5] 운영 배포 시 지켜야 할 체크리스트는 무엇인가요?")
print("[A5]", rag_chain.invoke("운영 배포 시 지켜야 할 체크리스트는 무엇인가요?"), "\n")

print("[Q6] 운영 DB에 접근하려면 어떤 절차가 필요한가요?")
print("[A6]", rag_chain.invoke("운영 DB에 접근하려면 어떤 절차가 필요한가요?"), "\n")

print("[Q7] 리프레시 휴가는 근속 몇 년마다, 며칠 주어지나요?")
print("[A7]", rag_chain.invoke("리프레시 휴가는 근속 몇 년마다, 며칠 주어지나요?"), "\n")

print("[Q8] 복지 제도 중 성장 지원과 건강 지원의 내용은 무엇인가요?")
print("[A8]", rag_chain.invoke("복지 제도 중 성장 지원과 건강 지원의 내용은 무엇인가요?"), "\n")

print("[Q9] 업무용 생성형 AI에 고객 데이터를 입력해도 되나요?")
print("[A9]", rag_chain.invoke("업무용 생성형 AI에 고객 데이터를 입력해도 되나요?"), "\n")

print("[Q10] 외부 협력사에 Drive 파일을 공유하려면 어떻게 해야 하나요?")
print("[A10]", rag_chain.invoke("외부 협력사에 Drive 파일을 공유하려면 어떻게 해야 하나요?"), "\n")

[Q1] 가상Tech의 유연근무 출근 시간 기준은 어떻게 되나요?
[A1] 가상Tech의 표준 근무시간은 평일 10:00-19:00이며, 핵심 협업시간은 11:00-16:00입니다. 유연근무에 대한 구체적인 기준은 문서에 명시되어 있지 않지만, 일반적으로 이 시간을 기준으로 업무를 진행해야 합니다. 

[Q2] 재택근무는 일주일에 몇 번, 어떻게 신청하나요?
[A2] 재택근무는 주 2회 가능하며, 매주 월요일까지 캘린더에 'WFH'로 표시하여 신청해야 합니다. 

[Q3] 회의 시간 기준(25분/50분)은 왜 정해져 있나요?
[A3] 회의 시간 기준(25분/50분)은 효율적인 회의 운영을 위해 정해져 있습니다. 짧은 시간 안에 핵심 안건을 논의하고 결정을 내리도록 하여 참석자들의 집중력을 유지하고, 불필요한 시간 낭비를 줄이기 위함입니다. 또한, 안건이 없는 회의는 거절할 수 있도록 하여 회의의 목적성을 더욱 강화하고 있습니다. 

[Q4] 신입사원 첫날에는 어떤 절차를 거쳐야 하나요?
[A4] 신입사원 첫날에는 다음 절차를 거쳐야 합니다:

1. HRIS 프로필 작성
2. 보안 교육 수강
3. MFA 등록
4. 팀 채널 입장
5. 버디와 30분 온보딩

모든 항목은 입사일 18:00까지 완료해야 합니다. 

[Q5] 운영 배포 시 지켜야 할 체크리스트는 무엇인가요?
[A5] 운영 배포 시 지켜야 할 체크리스트는 다음과 같습니다:

1. CI 테스트와 보안 스캔 통과 확인
2. 관측 지표 및 알람 준비 완료
3. 변경 공지 완료 확인
4. 배포 후 30분 동안 오류율, p95 지연시간, 핵심 전환율 관찰
5. 오류율이 기준 대비 2배 또는 핵심 API 성공률이 99.5% 미만일 경우 즉시 롤백 검토
6. 금요일 15:00 이후와 공휴일 전날에는 긴급 변경 외 운영 배포 금지

이 체크리스트를 준수하여 안정적인 운영 배포를 진행해야 합니다. 

[Q6] 운영 DB에 접근하려면 어떤 절차가 필요한가요?
[A6] 운영 DB에 접근하려면 다음 절차를 따라야 합니다:
